## Research paper summrizer

In [ ]:
langchain

In [10]:
# Research Paper Summarizer — Project Implementation
# Dependencies: pip install langchain langchain_community chromadb sentence-transformers transformers accelerate pypdf

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

In [12]:
# -----------------------
# CONFIG
# -----------------------
PDF_FILE = "research_paper.pdf"   

# -----------------------
# 1. Load & Split Paper
# -----------------------
print("Loading PDF...")
loader = PyPDFLoader(PDF_FILE)
pages = loader.load()
print(f"Loaded {len(pages)} pages.")

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = splitter.split_documents(pages)

# -----------------------
# 2. Create Embeddings & Vector Store
# -----------------------
print("Creating embeddings...")

from langchain_aws import BedrockEmbeddings
embeddings=BedrockEmbeddings(model_id='cohere.embed-english-v3', #amazon.titan-embed-text-v1
                        aws_access_key_id='',
                        aws_secret_access_key='',
                       region_name='us-east-1')

vectorstore = Chroma.from_documents(docs, embeddings)

# -----------------------
# 3. Load Hugging Face LLM
# -----------------------
from langchain_aws import ChatBedrockConverse
llm=ChatBedrockConverse(model='cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
llm.invoke("Hi").content
# -----------------------
# 4. Define Prompt Template
# -----------------------
template = """
You are an expert academic summarizer.

### Research Paper Chunk:
{paper_chunk}

### Instructions:
Summarize the key points in 3-4 bullet points. Focus on technical insights and contributions.
"""

prompt = PromptTemplate.from_template(template)
chain = prompt | llm | StrOutputParser()

# -----------------------
# 5. Run Summarization on Chunks
# -----------------------
print("Summarizing chunks...")
for i, doc in enumerate(docs[:3]):  # limit to first 3 chunks for demo
    summary = chain.invoke({"paper_chunk": doc.page_content})
    print(f"\n--- Summary {i+1} ---\n{summary}")


Loading PDF...
Loaded 5 pages.
Creating embeddings...
Summarizing chunks...

--- Summary 1 ---
- The paper proposes a clear distinction between "AI Agents" and "Agentic AI," which is crucial for understanding the scope and limitations of each in the field of Information Fusion, especially with the advent of Generative AI. 

- It offers a structured, conceptual taxonomy and application mapping of these two types of AI, highlighting their divergent design philosophies and providing clarity to the field. 

- The authors contribute a critical review, offering insights into the opportunities and challenges presented by each type of AI, which is essential for informed development and deployment decisions. 

- This work emphasizes the need for context awareness and a nuanced understanding of the capabilities and limitations of different AI systems, especially in multi-agent system environments.

--- Summary 2 ---
- The paper distinguishes between two AI paradigms: AI Agents and Agentic AI. AI

In [17]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os


# -----------------------
# CONFIG
# -----------------------
PDF_FILE = "research_paper.pdf"   
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

# -----------------------
# 1. Load & Split PDF
# -----------------------
loader = PyPDFLoader(PDF_FILE)
pages = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
docs = splitter.split_documents(pages)

# -----------------------
# 2. Load  LLM
# -----------------------

from langchain_aws import ChatBedrockConverse
llm=ChatBedrockConverse(model='cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
llm.invoke("Hi").content
# -----------------------
# 3. Define Prompt Template
# -----------------------
template = """
You are an expert assistant for summarizing academic and legal documents.

### Document Chunk:
{document_chunk}

### Instructions:
Extract and summarize the following sections:
1. Problem Statement
2. Proposed Method
3. Key Findings

Return the summary in JSON format with keys: "problem_statement", "proposed_method", "key_findings".
"""

prompt = PromptTemplate.from_template(template)
chain = prompt | llm | StrOutputParser()

# -----------------------
# 4. Run Summarization on Chunks
# -----------------------
print("Summarizing chunks...")
for i, doc in enumerate(docs[:3]):  # limit to first 3 chunks for demo
    summary = chain.invoke({"document_chunk": doc.page_content})
    print(f"\n--- Summary {i+1} ---\n{summary}")


Summarizing chunks...

--- Summary 1 ---
```json
{
  "problem_statement": "The problem addressed in this paper is the need to distinguish between 'AI Agents' and 'Agentic AI' in the field of information fusion, especially with the advancements in Generative AI. This differentiation is crucial as it impacts the design philosophies and applications of these technologies.",
  "proposed_method": {
    "taxonomy": "The paper proposes a conceptual taxonomy to clearly differentiate AI Agents from Agentic AI. AI Agents are defined as autonomous entities with agency, capable of acting and making decisions. On the other hand, Agentic AI refers to the broader ecosystem and infrastructure that enables AI Agents, including tools, platforms, and environments.",
    "application_mapping": "It offers an application mapping, exploring how the distinction between AI Agents and Agentic AI plays out in various domains, such as robotics, natural language processing, and cyber-physical systems.",
    "desig